### Step 1 — Download 2019 Yellow Taxi Data from NYC TLC

The original S3 bucket (nyc-tlc) is no longer publicly accessible.
We download directly from the official NYC TLC website using Python's urllib.

**We start with January 2019 only to validate before loading the full year.**

Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
Format: Parquet  
Destination: dbfs:/FileStore/nyc_taxi/raw/

### Step 1 — Set Up DBFS Directory Structure

Before downloading any data, we create the folder structure in DBFS.
This keeps raw, cleaned, and output data organized and separate.

> **Rule:** Raw data is never modified. We always write cleaned/transformed
> data to a separate folder so we can reprocess from raw at any time.

In [0]:
%sql
SHOW CATALOGS;

### Step 2 — Check Available Schemas in Workspace Catalog

Before creating a Volume, we need a schema (database) to put it in.
Let's see what already exists in the workspace catalog.

In [0]:
%sql
SHOW SCHEMAS IN workspace;

### Step 3 — Create Project Schema and Volume

We create a dedicated schema called `nyc_taxi` inside the `workspace` catalog.
Then we create a Volume inside it — this is where all our raw and cleaned data will live.

**Why a Volume?**  
Unity Catalog Volumes are the modern replacement for DBFS.
They are governed, auditable, and work with all Databricks features.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.nyc_taxi
COMMENT 'NYC Taxi Revenue Optimization Project - 2019 Data';

SHOW SCHEMAS IN workspace;

### Step 4 — Create a Volume

A Volume is a Unity Catalog object that provides governed access to
non-tabular files (parquet, csv, etc.) stored in cloud storage.

We create one Volume called `raw_data` inside `workspace.nyc_taxi`.
This is where we will land the downloaded parquet files before
converting them to Delta tables.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.nyc_taxi.raw_data
COMMENT 'Raw NYC TLC parquet files - do not modify';

SHOW VOLUMES IN workspace.nyc_taxi;

### Step 5 — Download January 2019 Data into Volume

We start with January 2019 only to validate the pipeline end-to-end
before committing to all 12 months.

**Why one month first?**
- Catches schema issues early
- Faster to debug on a small file (~130MB)
- Once January works, the rest is just a loop

Source: NYC TLC official CloudFront CDN (current as of 2024)
Destination: /Volumes/workspace/nyc_taxi/raw_data/

In [0]:
import urllib.request

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet"
dest = "/Volumes/workspace/nyc_taxi/raw_data/yellow_tripdata_2019-01.parquet"

print("⬇️  Downloading January 2019...")
urllib.request.urlretrieve(url, dest)
print("✅ Download complete:", dest)

### Step 6 — Verify the Downloaded File

Before loading into Spark, we confirm:
1. The file exists in the Volume
2. The file size is reasonable (~130MB for January)
3. Spark can actually read it and return a schema

In [0]:
files = dbutils.fs.ls("/Volumes/workspace/nyc_taxi/raw_data/")
for f in files:
    size_mb = round(f.size / (1024 * 1024), 2)
    print(f"📄 {f.name} — {size_mb} MB")

### Step 7 — Load Parquet into Spark and Inspect Schema

Now we read the parquet file into a Spark DataFrame.
This is a lazy operation — Spark will not scan the full file yet.

We are checking:
1. Column names and data types match what we expect
2. No obvious schema issues before we register it as a table

In [0]:
df_raw = spark.read.parquet("/Volumes/workspace/nyc_taxi/raw_data/yellow_tripdata_2019-01.parquet")

print(f"Columns: {len(df_raw.columns)}")
df_raw.printSchema()

### Step 8 — Column Assessment for Revenue Analysis

Not all 19 columns are useful. Here's what we keep and why:

| Column | Type | Why We Need It |
|---|---|---|
| `tpep_pickup_datetime` | timestamp | Extract hour, day of week, shift |
| `tpep_dropoff_datetime` | timestamp | Calculate trip duration |
| `PULocationID` | long | Pickup zone — key for zone analysis |
| `DOLocationID` | long | Dropoff zone — key for route analysis |
| `trip_distance` | double | Revenue per mile calculation |
| `fare_amount` | double | Base revenue |
| `tip_amount` | double | Critical — high tip zones = better shifts |
| `tolls_amount` | double | Affects net revenue |
| `congestion_surcharge` | double | Adds to total revenue |
| `total_amount` | double | Full revenue per trip |
| `passenger_count` | double | Data quality check |
| `RatecodeID` | double | Identifies airport runs vs standard |
| `payment_type` | long | Cash vs card affects tip patter

In [0]:
from pyspark.sql.functions import col, count, when, isnan

# Check nulls on columns we care about
key_cols = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID",
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "passenger_count", "RatecodeID"
]

null_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c) for c in key_cols
])

null_counts.show(vertical=True)

### Step 10 — Download All Data: 2019 + 2022 to 2026 April

**Strategy:**
- 2019: Pre-COVID baseline (12 months)
- 2022–2026 April: Post-COVID modern patterns (4 years + 4 months)
- Skipping 2020–2021: COVID anomaly years — not representative

**Total files:** 64 parquet files (~7–9GB estimated)
**Estimated download time:** 30–45 minutes

All files land in: /Volumes/workspace/nyc_taxi/raw_data/

In [0]:
import urllib.request
import os

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data"
dest_path = "/Volumes/workspace/nyc_taxi/raw_data"

# Define year/month ranges
download_targets = []

# 2019 — full year
for m in range(1, 13):
    download_targets.append((2019, m))

# 2022–2025 — full years
for y in range(2022, 2026):
    for m in range(1, 13):
        download_targets.append((y, m))

# 2026 — January to April only
for m in range(1, 5):
    download_targets.append((2026, m))

# Download loop
failed = []
skipped = []

for year, month in download_targets:
    filename = f"yellow_tripdata_{year}-{str(month).zfill(2)}.parquet"
    dest = f"{dest_path}/{filename}"
    url = f"{base_url}/{filename}"

    # Skip if already downloaded (e.g. January 2019)
    try:
        if dbutils.fs.ls(dest_path):
            existing = [f.name for f in dbutils.fs.ls(dest_path)]
            if filename in existing:
                print(f"⏭️  Skipping (exists): {filename}")
                skipped.append(filename)
                continue
    except:
        pass

    try:
        print(f"⬇️  Downloading {filename}...")
        urllib.request.urlretrieve(url, dest)
        print(f"✅ Done: {filename}")
    except Exception as e:
        print(f"❌ Failed: {filename} — {e}")
        failed.append(filename)

# Summary
total = len(download_targets)
print(f"\n--- Download Summary ---")
print(f"✅ Downloaded: {total - len(failed) - len(skipped)}")
print(f"⏭️  Skipped (already existed): {len(skipped)}")
print(f"❌ Failed: {len(failed)}")
if failed:
    print(f"Failed files: {failed}")

### Step 11 — Verify All Downloaded Files

Before loading into Spark, we confirm:
1. Total file count matches expectation (59 files)
2. No file is suspiciously small (corrupt or partial download)
3. All years are represented

A healthy parquet file for one month should be at least 20MB.
Anything under 5MB is a red flag.

In [0]:
files = dbutils.fs.ls("/Volumes/workspace/nyc_taxi/raw_data/")

total_size = 0
small_files = []

print(f"{'File':<45} {'Size (MB)':>10}")
print("-" * 57)

for f in sorted(files, key=lambda x: x.name):
    size_mb = round(f.size / (1024 * 1024), 2)
    total_size += size_mb
    flag = " ⚠️ TOO SMALL" if size_mb < 5 else ""
    print(f"{f.name:<45} {size_mb:>10.2f} MB{flag}")
    if size_mb < 5:
        small_files.append(f.name)

print("-" * 57)
print(f"{'Total files:':<45} {len(files):>10}")
print(f"{'Total size:':<45} {round(total_size/1024, 2):>10.2f} GB")

if small_files:
    print(f"\n⚠️  Suspicious files: {small_files}")
else:
    print("\n✅ All files look healthy")

### Step 12 — Schema Consistency Check Across All Years

This is critical before building the Delta table.
TLC has changed column names and data types across years.
If schemas don't match, our wildcard read will either fail or
silently produce nulls — both are bad.

We read each file's schema independently and compare against
January 2019 as our baseline. Any differences get flagged.

In [0]:
from pyspark.sql.functions import col

path = "/Volumes/workspace/nyc_taxi/raw_data"
files = sorted([f.name for f in dbutils.fs.ls(path) if f.name.endswith(".parquet")])

# Use January 2019 as baseline
baseline = spark.read.parquet(f"{path}/yellow_tripdata_2019-01.parquet")
baseline_schema = {f.name: str(f.dataType) for f in baseline.schema.fields}

print(f"Baseline columns (2019-01): {len(baseline_schema)}\n")

mismatches = {}

for filename in files:
    if filename == "yellow_tripdata_2019-01.parquet":
        continue
    
    df = spark.read.parquet(f"{path}/{filename}")
    file_schema = {f.name: str(f.dataType) for f in df.schema.fields}
    
    issues = []
    
    # Check for missing columns
    for col_name in baseline_schema:
        if col_name not in file_schema:
            issues.append(f"MISSING: {col_name}")
    
    # Check for new columns
    for col_name in file_schema:
        if col_name not in baseline_schema:
            issues.append(f"NEW COL: {col_name}")
    
    # Check for type changes
    for col_name in baseline_schema:
        if col_name in file_schema and baseline_schema[col_name] != file_schema[col_name]:
            issues.append(f"TYPE CHANGE: {col_name} — {baseline_schema[col_name]} → {file_schema[col_name]}")
    
    if issues:
        mismatches[filename] = issues
    else:
        print(f"✅ {filename}")

if mismatches:
    print("\n⚠️  Schema Mismatches Found:")
    for fname, issues in mismatches.items():
        print(f"\n❌ {fname}")
        for issue in issues:
            print(f"   → {issue}")
else:
    print("\n✅ All schemas match baseline")

### Step 13 — Define Unified Target Schema and Load All Files

We now read all 59 files with a standardized schema applied on read.

**Fixes applied:**
1. Cast all numeric IDs to consistent types (Long vs Integer differences)
2. Rename `Airport_fee` → `airport_fee` for 2023+ files
3. Cast `airport_fee` to DoubleType across all years
4. Add `cbd_congestion_fee` as nullable Double (null for years before 2025)
5. Add `data_year` and `data_month` columns for partitioning

**Result:** One unified DataFrame ready to write to Delta.

In [0]:
from pyspark.sql.functions import col, lit, year, month, to_timestamp
from pyspark.sql.types import LongType, DoubleType, IntegerType, StringType
from functools import reduce

path = "/Volumes/workspace/nyc_taxi/raw_data"
files = sorted([f.name for f in dbutils.fs.ls(path) if f.name.endswith(".parquet")])

dfs = []

for filename in files:
    df = spark.read.parquet(f"{path}/{filename}")
    
    # Fix 1: Rename Airport_fee → airport_fee if needed (2023+)
    if "Airport_fee" in df.columns:
        df = df.withColumnRenamed("Airport_fee", "airport_fee")
    
    # Fix 2: Add airport_fee if completely missing
    if "airport_fee" not in df.columns:
        df = df.withLiteral(lit(None).cast(DoubleType()).alias("airport_fee"))
    
    # Fix 3: Add cbd_congestion_fee if missing (pre-2025 files)
    if "cbd_congestion_fee" not in df.columns:
        df = df.withColumn("cbd_congestion_fee", lit(None).cast(DoubleType()))
    
    # Fix 4: Standardize all column types to match baseline
    df = df \
        .withColumn("VendorID",        col("VendorID").cast(LongType())) \
        .withColumn("passenger_count", col("passenger_count").cast(DoubleType())) \
        .withColumn("RatecodeID",      col("RatecodeID").cast(DoubleType())) \
        .withColumn("PULocationID",    col("PULocationID").cast(LongType())) \
        .withColumn("DOLocationID",    col("DOLocationID").cast(LongType())) \
        .withColumn("airport_fee",     col("airport_fee").cast(DoubleType()))

    # Fix 5: Add partitioning columns from filename
    yr  = int(filename.split("_")[2].split("-")[0])
    mo  = int(filename.split("_")[2].split("-")[1].replace(".parquet",""))
    df = df.withColumn("data_year",  lit(yr)) \
           .withColumn("data_month", lit(mo))

    dfs.append(df)
    print(f"✅ Staged: {filename}")

# Union all into one DataFrame
df_all = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=False), dfs)

print(f"\n✅ All files staged successfully")
print(f"Total columns: {len(df_all.columns)}")
print(f"Columns: {df_all.columns}")

### Step 14 — Write to Delta Table (Partitioned by Year)

We now write the unified DataFrame to a Delta table partitioned by `data_year`.

**Why partition by year?**
- Queries filtered by year (e.g. 2019 only) scan only that partition
- Dramatically faster for our comparative analysis
- Delta can skip entire year folders when not needed

**Expected write time:** 10–20 minutes for 3.67GB across 59 files.
Do not interrupt this cell once it starts.

In [0]:
(df_all
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("data_year")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.nyc_taxi.yellow_trips_raw")
)

print("✅ Delta table written: workspace.nyc_taxi.yellow_trips_raw")

### Step 15 — Verify Delta Table

Final checks before we close the ingestion notebook:
1. Total row count across all years
2. Row count per year — confirms no year got dropped
3. Column count and schema intact
4. Delta table properties — confirms partitioning is in place

In [0]:
%sql
SELECT 
    data_year,
    COUNT(*) AS total_trips,
    ROUND(COUNT(*) / 1000000.0, 2) AS trips_millions
FROM workspace.nyc_taxi.yellow_trips_raw
GROUP BY data_year
ORDER BY data_year;

### Step 16 — Final Table Verification

Confirm schema, partitions, and Delta properties are all correct
before closing the ingestion notebook.

In [0]:
%sql
-- Confirm schema
DESCRIBE TABLE workspace.nyc_taxi.yellow_trips_raw;

In [0]:
%sql
DESCRIBE DETAIL workspace.nyc_taxi.yellow_trips_raw;